# Step 1A — Pick a Theme

The child picks a story theme — either a preset tile or a custom text input.

```
Character selection
    └─ Theme input (preset tile or custom text)
           ├─ POST /api/check-theme   → SAFE / UNSAFE classification
           └─ POST /api/story-opening → opening text (in character language) + first illustration
```

---
## Setup

In [ ]:
import sys, os, base64
sys.path.insert(0, '../backend')

from dotenv import load_dotenv
load_dotenv('../backend/.env')

print('PROJECT:      ', os.environ.get('GOOGLE_CLOUD_PROJECT', '❌ not set'))
print('IMAGE_MODEL:  ', os.environ.get('IMAGE_MODEL', '❌ not set'))
print('GEMINI_API_KEY:', 'set' if os.environ.get('GEMINI_API_KEY') else '❌ not set')

---
## Initial Step — Character Selection

The child picks a storyteller character before anything else. Each character has a unique voice, language, and image art style.

In [ ]:
from characters import CHARACTERS, get_character

print(f'{len(CHARACTERS)} characters available:\n')
for c in CHARACTERS.values():
    print(f'  {c.id:20s}  lang={c.language:10s}  voice={c.voice_name:15s}  {c.name}')

In [ ]:
# Pick a character
char = get_character('wizard')

print(f'Name:         {char.name}')
print(f'Language:     {char.language}')
print(f'Voice:        {char.voice_name}')
print(f'Image style:  {char.image_style}')
print(f'\n--- System prompt (first 400 chars) ---')
print(char.system_prompt[:400], '...')

---
## Preset Themes

These are the exact theme tiles shown in the app. The child taps one — no typing needed.

In [ ]:
THEMES = [
    ('🦁', 'Animals'),
    ('🚀', 'Space'),
    ('🏰', 'Kingdoms'),
    ('🌊', 'Ocean'),
    ('🍕', 'Food'),
    ('🌳', 'Jungle'),
    ('🦄', 'Magic'),
    ('🤖', 'Robots'),
    ('🎃', 'Spooky'),
    ('🏔️', 'Adventure'),
    ('🎪', 'Circus'),
    ('🦕', 'Dinosaurs'),
]

LIFE_SKILLS_THEMES = [
    ('🤝', 'Sharing'),
    ('💪', 'Courage'),
    ('🙏', 'Gratitude'),
    ('🎨', 'Creativity'),
    ('🌍', 'Kindness'),
]

print('Story themes:')
for emoji, label in THEMES:
    print(f'  {emoji}  {label}')

print('\nLife skills themes:')
for emoji, label in LIFE_SKILLS_THEMES:
    print(f'  {emoji}  {label}')

---
## Theme Input

Set `THEME` to any label from the lists above, or enter a custom string.  
Preset labels skip the safety check (pre-approved in the app). Custom strings go through it.

In [ ]:
# Pick a preset label or type a custom theme
THEME = 'Space'                                      # preset
# THEME = 'Kindness'                                 # life skills preset
# THEME = 'space adventure with friendly aliens'     # custom

---
## `POST /api/check-theme` — Safety Check

Flash Lite classifies the theme as `SAFE` or `UNSAFE` before anything is generated.  
If unsafe, the frontend shows an error and the child is asked to pick a different theme.

In [ ]:
from image_gen import _is_safe_for_children, EXTRACT_MODEL

print(f'Theme:  {THEME!r}')
print(f'Model:  {EXTRACT_MODEL}')

safe = await _is_safe_for_children(THEME)

if safe:
    print('\n✅ SAFE — proceeding to story opening')
else:
    print('\n❌ UNSAFE — frontend would reject this theme')

In [ ]:
# Batch-check all preset labels + some custom strings
import asyncio

to_check = [label for _, label in THEMES + LIFE_SKILLS_THEMES] + [
    'fighting with guns',
    'haunted house horror',
    'my favourite toy comes to life',
]

results = await asyncio.gather(*[_is_safe_for_children(t) for t in to_check])

for theme, is_safe in zip(to_check, results):
    status = '✅ SAFE  ' if is_safe else '❌ UNSAFE'
    print(f'{status}  {theme}')

---
## `POST /api/story-opening` — Opening Text + First Illustration

Once the theme is confirmed, this runs as `StoryScreen` mounts — before the child clicks Begin.

1. Flash Lite generates the opening (story text in the character's language + English scene description)
2. Image model renders the first illustration from the scene description

The image is held in `prewarmImageRef` and shown only when the child clicks **Begin Story**.

In [ ]:
from image_gen import _generate_opening

print(f'Character: {char.name} (language: {char.language})')
print(f'Theme:     {THEME!r}')
print(f'Model:     {EXTRACT_MODEL}\n')

opening_text, scene_description = await _generate_opening(
    character_name=char.name,
    character_language=char.language,
    theme=THEME,
    prop_description='',
)

print('=== STORY OPENING (in character language) ===')
print(opening_text)
print('\n=== SCENE DESCRIPTION (English, for image gen) ===')
print(scene_description)

In [ ]:
from image_gen import IMAGE_MODEL, SAFETY_PREFIX, _generate_imagen, _generate_gemini, _generate_gemini_api_key, _api_key_client
from IPython.display import Image, display

prompt = f'{SAFETY_PREFIX}{char.image_style}, {scene_description}'

print(f'Image model: {IMAGE_MODEL}')
print(f'Prompt:      ...{char.image_style[:50]}, {scene_description[:80]}...')
print()

if IMAGE_MODEL.startswith('imagen-'):
    image_b64, mime_type = await _generate_imagen(prompt)
elif _api_key_client:
    image_b64, mime_type = await _generate_gemini_api_key(prompt)
else:
    image_b64, mime_type = await _generate_gemini(prompt)

print(f'Image: {mime_type}, {len(image_b64):,} chars (base64)')
display(Image(data=base64.b64decode(image_b64)))

In [ ]:
# Full endpoint — wraps both steps above in one call
from image_gen import generate_story_opening, StoryOpeningRequest

request = StoryOpeningRequest(
    character_id=char.id,
    character_name=char.name,
    character_language=char.language,
    image_style=char.image_style,
    theme=THEME,
    prop_description='',
)

result = await generate_story_opening(request)

print('Opening text:')
print(result.opening_text)
print(f'\nScene: {result.scene_description}')
print(f'Image: {result.mime_type}, {len(result.image_data):,} chars')
display(Image(data=base64.b64decode(result.image_data)))